In [ ]:
# ================= build the per-frame phi/psi cache for persist50 coil (same format as the persist90 cache) =================
# Source: per-frame phi/psi of every gated residue (all 1001 frames); residue labels and MD persistence fractions
# from persist90_ss_burial_simple.csv. Coil = static ss_class "coil"; persist = ss_persist and burial_persist >= cut.
# With PERSIST_CUT = 0.9 this reproduces md_phipsi_perframe_coil_persist90_panav10seedens.npz exactly (824 residues).
import os
import numpy as np
import pandas as pd

FIG4 = "../../data/analysis/Fig4"
PERSIST_CUT = 0.5
DROP = 100                                     # equilibration frames dropped (same as the persist90 cache)
SRC_NPZ = f"{FIG4}/per_protein_analysis/abnormal_coil/md_dssp8_phipsi_gated_allframes_panav10seedens.npz"
PERSIST_CSV = f"{FIG4}/per_protein_analysis/persist90_ss_burial_simple.csv"
OUT_NPZ = (f"{FIG4}/per_protein_analysis/abnormal_coil/"
           f"md_phipsi_perframe_coil_persist{int(100 * PERSIST_CUT)}_panav10seedens.npz")

src = np.load(SRC_NPZ)
res = pd.DataFrame(dict(entry=src["entry"].astype(str), resnum=src["resnum"], col=np.arange(len(src["entry"]))))
res = res.merge(pd.read_csv(PERSIST_CSV), on=["entry", "resnum"], how="left")
keep = res[(res.ss_class == "coil") & res.exposure.isin(["buried", "exposed"])
           & (res.ss_persist >= PERSIST_CUT) & (res.burial_persist >= PERSIST_CUT)]
keep = keep.sort_values(["entry", "resnum"]).reset_index(drop=True)          # same residue order as the persist90 cache

phi = src["phi"][DROP:, keep.col.to_numpy()].T.astype(np.float32)          # [residue, frame]
psi = src["psi"][DROP:, keep.col.to_numpy()].T.astype(np.float32)
assert np.isfinite(phi).all() and np.isfinite(psi).all()
np.savez_compressed(OUT_NPZ, entry=keep.entry.to_numpy().astype(str), resnum=keep.resnum.to_numpy(),
                    exposure=keep.exposure.to_numpy().astype(str), phi=phi.ravel(), psi=psi.ravel(),
                    owner=np.repeat(np.arange(len(keep), dtype=np.int32), phi.shape[1]))
print(f"persist{int(100 * PERSIST_CUT)} coil: {keep.exposure.value_counts().to_dict()}, "
      f"{phi.shape[1]} frames each -> {os.path.basename(OUT_NPZ)}")
